# Clasificación: Naive Bayes y KNN

**Datos:** versión versionada en el repositorio (`origin/main`), sin cambios locales.

- `outputs/transformed/X_nhanes.csv` → 48 710 × 42
- `outputs/transformed/y_nhanes.csv` → `Disease_Indicator`

**Aviso metodológico:** en esta versión la etiqueta del repositorio es
`diabetes OR obesidad (IMC ≥ 30) OR colesterol ≥ 200`, e incluye menores.
Nosotros la usamos tal como estaba en el repositorio; por eso la prevalencia es 41,1 % y no 12 %.

**Pregunta del taller:** ¿nos funciona un solo modelo, los dos, o cómo decidimos?
Respuesta corta: comparamos los dos en el mismo tablero y al final recomendamos
**uno principal + uno como línea base**.

In [1]:
from pathlib import Path
import time

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

raiz = Path.cwd()
if not (raiz / "outputs/transformed/X_nhanes.csv").exists():
    raiz = raiz.parent
X = pd.read_csv(raiz / "outputs/transformed/X_nhanes.csv")
y = pd.read_csv(raiz / "outputs/transformed/y_nhanes.csv")["Disease_Indicator"]

print("X:", X.shape, "| y:", y.shape)
print("Clases:", y.value_counts().to_dict())
print("Prevalencia:", round(float(y.mean()), 4))

X: (48710, 42) | y: (48710,)
Clases: {0: 28687, 1: 20023}
Prevalencia: 0.4111


## 1. Tablero común

Misma partición para los dos modelos: `test_size=0.25`, `random_state=42`,
`stratify=y`. Sin esto, la comparación no vale.

Nota: usamos los archivos ya generados por el proyecto y no rehacemos el escalado
ni el PCA originales; solo creamos la partición para modelar.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Prevalencia train:", round(float(y_train.mean()), 4),
      "| test:", round(float(y_test.mean()), 4))

Train: (36532, 42) | Test: (12178, 42)
Prevalencia train: 0.4111 | test: 0.4111


## 2. Escalado correcto para KNN

Las 10 numéricas ya vienen escaladas del repositorio; las 32 dummies están en 0/1.
Si aplicáramos `StandardScaler` a **todo**, las dummies raras se inflarían y
dominarían la distancia. Por eso escalamos solo lo numérico con `ColumnTransformer`
y dejamos las dummies intactas. Naive Bayes no necesita este paso.

In [3]:
NUMERICAS = ["RIDAGEYR", "BMXWT", "BMXHT", "BMXWAIST",
             "DR1TKCAL", "DR1TSUGR", "DR1TPROT", "DR1TCARB", "DR1TFIBE", "DR1TTFAT"]

def preparador():
    return ColumnTransformer(
        transformers=[("num", StandardScaler(), NUMERICAS)],
        remainder="passthrough",
    )

def medir(modelo,Xt, yt):
    t0 = time.perf_counter()
    modelo.fit(X_train, y_train)
    tiempo = time.perf_counter() - t0
    filas = {"tiempo_fit_s": tiempo}
    for nombre, Xm, ym in (("train", X_train, y_train), ("test", Xt, yt)):
        pred = modelo.predict(Xm)
        prob = modelo.predict_proba(Xm)[:, 1]
        filas[f"accuracy_{nombre}"] = accuracy_score(ym, pred)
        filas[f"precision_{nombre}"] = precision_score(ym, pred, zero_division=0)
        filas[f"recall_{nombre}"] = recall_score(ym, pred, zero_division=0)
        filas[f"f1_{nombre}"] = f1_score(ym, pred, zero_division=0)
        filas[f"roc_auc_{nombre}"] = roc_auc_score(ym, prob)
    return filas, modelo

modelos = {
    "GaussianNB": GaussianNB(),
    "KNN k=5": Pipeline([("prep", preparador()),
                         ("knn", KNeighborsClassifier(n_neighbors=5))]),
}

resultados, ajustados = [], {}
for nombre, modelo in modelos.items():
    filas, ajustado = medir(modelo, X_test, y_test)
    filas["modelo"] = nombre
    resultados.append(filas)
    ajustados[nombre] = ajustado

tabla = pd.DataFrame(resultados).set_index("modelo")
tabla[["accuracy_train", "accuracy_test", "precision_test",
       "recall_test", "f1_test", "roc_auc_test", "tiempo_fit_s"]].round(3)

,accuracy_train,accuracy_test,precision_test,recall_test,f1_test,roc_auc_test,tiempo_fit_s
modelo,,,,,,,
GaussianNB,0.758,0.762,0.655,0.889,0.754,0.856,0.018
KNN k=5,0.860,0.799,0.757,0.753,0.755,0.873,0.016


### Lectura

- **Accuracy engaña si solo se mira train**: KNN k=5 saca 0,860 en train pero 0,799 en test.
- NB detecta más positivos (recall 0,889) pero con mucha más falsa alarma (precisión 0,655).
- KNN k=5 es más preciso (0,757) pero deja escapar más positivos (recall 0,753).
- Por eso no decidimos el veredicto con accuracy: usamos **F1 + recall**.

## 3. Explorar k

Probamos impares 1, 5, 11, 21, 51 y además `weights="distance"` con el mejor k.
Recordatorio: con `weights="distance"` el accuracy en train sale 1,000 porque cada
punto se tiene a sí mismo a distancia 0; **no es señal de acierto**, es un artefacto.

In [4]:
barrido = []
for k in [1, 5, 11, 21, 51]:
    filas, _ = medir(Pipeline([("prep", preparador()),
                               ("knn", KNeighborsClassifier(n_neighbors=k))]),
                     X_test, y_test)
    barrido.append({"k": k, "weights": "uniform",
                    "accuracy_train": filas["accuracy_train"],
                    "accuracy_test": filas["accuracy_test"],
                    "f1_test": filas["f1_test"],
                    "recall_test": filas["recall_test"]})

mejor_k = int(max(barrido, key=lambda r: r["f1_test"])["k"])
filas, _ = medir(Pipeline([("prep", preparador()),
                           ("knn", KNeighborsClassifier(n_neighbors=mejor_k,
                                                        weights="distance"))]),
                 X_test, y_test)
barrido.append({"k": mejor_k, "weights": "distance",
                "accuracy_train": filas["accuracy_train"],
                "accuracy_test": filas["accuracy_test"],
                "f1_test": filas["f1_test"],
                "recall_test": filas["recall_test"]})

tabla_k = pd.DataFrame(barrido)
print("Mejor k por F1 en test:", mejor_k)
tabla_k.round(3)

Mejor k por F1 en test: 51


,k,weights,accuracy_train,accuracy_test,f1_test,recall_test
0,1,uniform,1.000,0.757,0.699,0.685
1,5,uniform,0.860,0.799,0.755,0.753
2,11,uniform,0.843,0.813,0.773,0.776
3,21,uniform,0.837,0.818,0.782,0.792
4,51,uniform,0.830,0.825,0.792,0.811
5,51,distance,1.000,0.826,0.793,0.811


## 4. Matrices de confusión (test)

In [5]:
filas = []
for nombre, modelo in ajustados.items():
    tn, fp, fn, tp = confusion_matrix(y_test, modelo.predict(X_test)).ravel()
    filas.append({"modelo": nombre, "TN": tn, "FP": fp, "FN": fn, "TP": tp})
pd.DataFrame(filas).set_index("modelo")

,TN,FP,FN,TP
modelo,,,,
GaussianNB,4826,2346,555,4451
KNN k=5,5964,1208,1237,3769


## 5. Veredicto (máximo 8 líneas)

**Recomendamos KNN con k=51 y pesos uniformes como modelo principal**, y
**GaussianNB como línea base permanente**.

- KNN k=51: accuracy test **0,825**, F1 **0,792**, recall **0,782**; mejora estable
  desde k=1 (0,757) sin el artefacto de `weights="distance"` en train.
- GaussianNB: accuracy **0,762**, F1 **0,754**, recall **0,889**; detecta más casos
  positivos, pero con 2 346 falsos positivos (NB) frente a 1 208 de KNN k=5.
- Descartamos k=1 (memoriza: 1,000 train / 0,757 test) y k=5 (aún nervioso).
- `weights="distance"` da F1 0,793, apenas por encima; su 1,000 en train es un
  artefacto, no una ventaja real.
- Conservamos NB porque entrena en ~0,02 s y sirve para detectar si el problema
  se complica; sus probabilidades no las usamos como cifras oficiales.
- Tomamos la decisión con tres criterios: F1/recall (KNN), estabilidad (k grande),
  y velocidad/línea base (NB).

## 6. Limitaciones que confesamos

- La etiqueta del repositorio mezcla diabetes, obesidad y colesterol; no es solo diabetes.
- La base incluye menores y no usa pesos de encuesta.
- Hay 32 dummies con multicolinealidad; GaussianNB cuenta evidencia duplicada.
- KNN paga el costo en `predict()`, no en `fit()`.